# Capstone W4 All Functions v9


This code generates predictions for the ICL ML/AI Capstone BBO Challenge.

In order to run successfully, the following requirements must be met:

- the input and output files for each function must be present in a /data subfolder
- the input and output files must be Numpy arrays
- the files must be called week_n_inputs.npy and week_n_outputs.npy where n is the week prior to that for which you wish to make predictions. For example, in Week 5 of the challenge n = 4
- you must set the `functionID` and `week` global variables below

In [ ]:
###
##
## This is to select next BO point to sample for the ICL ML/AI captone project
## Set Function# and Week# each week
## It assumes we have already appended last weeks's queries and results to the x and y input files and stored them in the /data folder for this new week
##
###
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt

## 1. Edit the global variables

- the functionID parameter must be set to the number of the function you wish to predict. For example, if you wish to predict F7, set functionID = 7
- the week parameter must be set to the week for which you wish to predict. For example, if you wish to predict for Week 5, set functionID = 5
- optionally, the `verbose` parameter can be set to True or False. Set it to true if you want console output logging the progress of the prediction process

In [ ]:
functionID = 8
week = 4
np.random.seed(42)
verbose = False

Set the filenames for the input and output files to be used
 
- The input file is the sample points to date for this function
- The output file is the results to date for this function

Both are used as input to the prediction process

In [ ]:
##  Don't edit this ##
xfile = "C:/users/ianma/module0_capstone/week_" + str(week) + "/data/f" + str(functionID) + "_w" + str(week) + "_inputs.npy"
yfile = "C:/users/ianma/module0_capstone/week_" + str(week) + "/data/f" + str(functionID) + "_w" + str(week) + "_outputs.npy"

# 2. Edit the function-specific parameters if required

These include bounds, dimensions and acquisition variables


In [ ]:
if functionID == 1:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0], [1, 1]])
    dimensions = 2
#
if functionID == 2:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0], [1, 1]])
    dimensions = 2
#
if functionID == 3:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0], [1, 1, 1]])
    dimensions = 3
#
if functionID == 4:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0, 0], [1, 1, 1, 1]])
    dimensions = 4
#
if functionID == 5:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0, 0], [1, 1, 1, 1]])
    dimensions = 4
#
if functionID == 6:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0, 0, 0], [1, 1, 1, 1, 1]])
    dimensions = 5
#
if functionID == 7:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1]])
    dimensions = 6
#
if functionID == 8:
    aqFunc = 'ei'
    kappa = 20
    xi = 0.1
    bounds = np.array([[0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1]])
    dimensions = 8

# 3. Import the data

In [ ]:
X = np.load(xfile)
Y = np.load(yfile)
Y_best = Y.max()


# 4. Fit the Gaussian Process

In [ ]:
kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gp.fit(X, Y)

# 5a. Define the Upper Confidence Bound (UCB) Acquisition Function

In [ ]:
def ucbAcquisition(X_cand, gp, kappa):
    mean, std = gp.predict(X_cand, return_std=True)
    return mean + kappa * std  # We want to maximize this value

# 5b. Define the Expected Improvement (EI) Acquisition Function

In [ ]:
def eiAcquisition(X_cand, gp, Y_best, xi):
    mean, std = gp.predict(X_cand, return_std=True)
    with np.errstate(divide='warn'):
        improvement = mean - Y_best - xi
        Z = improvement / std
        ei = improvement * norm.cdf(Z) + std * norm.pdf(Z)
        ei[std == 0.0] = 0.0
    return ei

# 6. Define a function to generate and return the next sample point

In [ ]:
def propose_next_point(gp, bounds, dimensions, n_candidates=10000):
    # Randomly sample candidates in nD space
    candidates = np.random.uniform(bounds[0], bounds[1], size=(n_candidates, dimensions))
    # Call the relevant acquisition function
    if aqFunc == 'ucb':
        scores = ucbAcquisition(candidates, gp, kappa)
        if verbose:
            print('\nAq Function: UCB', '\nY_best:', Y_best, '\nKappa:',  kappa)
    if aqFunc == 'ei':
        if verbose:
            print('\nAq Function: EI', '\nY_best:', Y_best, '\nXi:',  xi)
        scores = eiAcquisition(candidates, gp, Y_best, xi)
        
    # Return the candidate with the highest score
    next_point = candidates[np.argmax(scores)]
    
    if verbose:
        print('Winning candidate number: ', np.argmax(scores))
        print('Next point pre-decimal adjustment:', next_point)

    next_point = np.round(next_point,6)
    return next_point

# 7. Main Function - request the next sample point

In [ ]:
next_point = propose_next_point(gp, bounds, dimensions)

# 8. Create and print the query string for submission

In [ ]:
columns= len(next_point)
colcount = 0
query=''
packzero = ''
while colcount < columns:
    element = str(round(next_point[colcount], 6))
    strlen = len(element)
    if strlen < 8:
        packlen = 8 - strlen
        while len(packzero) < packlen:
            packzero = packzero + '0'
        element = element + packzero
    query = query + '-' + element
    colcount = colcount + 1

#8. Print the query string
print(f"Function {functionID}")
print (query[1:])